In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


# ============================================================
# 1. CONFIGURATION
# ============================================================

storage_account = "stsupplychain2026xx"

silver_events_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/delivery_events/"
)

silver_shipments_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipments/"
)

shipment_lifecycle_path = (
    f"abfss://logistics@{storage_account}.dfs.core.windows.net/"
    "silver/shipment_lifecycle/"
)


# ============================================================
# 2. LECTURE DES TABLES SILVER
# ============================================================

silver_events = (
    spark.read
    .format("delta")
    .load(silver_events_path)
)

silver_shipments = (
    spark.read
    .format("delta")
    .load(silver_shipments_path)
)

print(
    "Nombre de shipments :",
    silver_shipments.count()
)

print(
    "Nombre d'events :",
    silver_events.count()
)


# ============================================================
# 3. CRÉER LES MILESTONES DU LIFECYCLE
# ============================================================

lifecycle_milestones = (
    silver_events
    .groupBy("shipment_id")
    .agg(

        # Nombre total d'événements
        F.count("*").alias("event_count"),

        # Nombre de types d'événements différents
        F.countDistinct(
            "event_status"
        ).alias("distinct_status_count"),

        # Première apparition de CREATED
        F.min(
            F.when(
                F.col("event_status") == "CREATED",
                F.col("event_time")
            )
        ).alias("created_at"),

        # Première apparition de PACKED
        F.min(
            F.when(
                F.col("event_status") == "PACKED",
                F.col("event_time")
            )
        ).alias("packed_at"),

        # Première apparition de SHIPPED
        F.min(
            F.when(
                F.col("event_status") == "SHIPPED",
                F.col("event_time")
            )
        ).alias("shipped_at"),

        # Première apparition de HUB_SCAN
        F.min(
            F.when(
                F.col("event_status") == "HUB_SCAN",
                F.col("event_time")
            )
        ).alias("hub_scan_at"),

        # Première apparition de OUT_FOR_DELIVERY
        F.min(
            F.when(
                F.col("event_status") == "OUT_FOR_DELIVERY",
                F.col("event_time")
            )
        ).alias("out_for_delivery_at"),

        # Première apparition de DELIVERED
        F.min(
            F.when(
                F.col("event_status") == "DELIVERED",
                F.col("event_time")
            )
        ).alias("delivered_at"),

        # Premier événement métier
        F.min(
            "event_time"
        ).alias("first_event_time"),

        # Dernier événement métier
        F.max(
            "event_time"
        ).alias("last_event_time"),

        # Plus grande étape atteinte
        F.max(
            "event_sequence"
        ).alias("max_event_sequence")
    )
)


# ============================================================
# 4. TROUVER LE STATUT ACTUEL
# ============================================================

latest_event_window = (
    Window
    .partitionBy("shipment_id")
    .orderBy(
        F.col("event_time").desc(),
        F.col("updated_at").desc(),
        F.col("ingestion_timestamp").desc()
    )
)

latest_events = (
    silver_events

    .withColumn(
        "_row_number",
        F.row_number().over(
            latest_event_window
        )
    )

    .filter(
        F.col("_row_number") == 1
    )

    .select(
        "shipment_id",

        F.col(
            "event_status"
        ).alias("current_status"),

        F.col(
            "event_time"
        ).alias("current_status_time"),

        F.col(
            "event_sequence"
        ).alias("current_event_sequence")
    )
)


# ============================================================
# 5. CONSTRUIRE LE LIFECYCLE COMPLET
# ============================================================

shipment_lifecycle = (
    silver_shipments

    .select(
        "shipment_id",
        "order_id",
        "carrier_id",
        "shipment_time",
        "expected_delivery_time",
        "origin_city",
        "destination_city",
        "distance_km"
    )

    .join(
        lifecycle_milestones,
        on="shipment_id",
        how="left"
    )

    .join(
        latest_events,
        on="shipment_id",
        how="left"
    )
)


# ============================================================
# 6. INDICATEURS DE LIFECYCLE
# ============================================================

shipment_lifecycle = (
    shipment_lifecycle

    # Un shipment est livré si un event DELIVERED existe
    .withColumn(
        "is_delivered",
        F.col("delivered_at").isNotNull()
    )

    # Lifecycle complet :
    # les 6 étapes attendues sont présentes
    .withColumn(
        "is_lifecycle_complete",
        (
            F.col("created_at").isNotNull()
            & F.col("packed_at").isNotNull()
            & F.col("shipped_at").isNotNull()
            & F.col("hub_scan_at").isNotNull()
            & F.col("out_for_delivery_at").isNotNull()
            & F.col("delivered_at").isNotNull()
        )
    )

    # Durée totale du lifecycle en heures
    .withColumn(
        "delivery_duration_hours",

        F.when(
            F.col("delivered_at").isNotNull(),

            (
                F.unix_timestamp(
                    "delivered_at"
                )
                -
                F.unix_timestamp(
                    "shipment_time"
                )
            ) / 3600
        )
    )

    # Retard par rapport à la date prévue
    .withColumn(
        "delivery_delay_hours",

        F.when(
            F.col("delivered_at").isNotNull(),

            (
                F.unix_timestamp(
                    "delivered_at"
                )
                -
                F.unix_timestamp(
                    "expected_delivery_time"
                )
            ) / 3600
        )
    )
)


# ============================================================
# 7. ÉCRITURE DELTA
# ============================================================

(
    shipment_lifecycle
    .write
    .format("delta")
    .mode("overwrite")
    .save(shipment_lifecycle_path)
)

print(
    "Shipment lifecycle construit avec succès."
)


# ============================================================
# 8. AFFICHAGE
# ============================================================

display(
    shipment_lifecycle
    .select(
        "shipment_id",
        "order_id",
        "current_status",
        "current_status_time",
        "created_at",
        "packed_at",
        "shipped_at",
        "hub_scan_at",
        "out_for_delivery_at",
        "delivered_at",
        "event_count",
        "is_delivered",
        "is_lifecycle_complete",
        "delivery_duration_hours",
        "delivery_delay_hours"
    )
    .orderBy("shipment_id")
)

In [0]:
from pyspark.sql import functions as F

print("=== Vérification S0131 ===")

display(
    shipment_lifecycle
    .filter(
        F.col("shipment_id") == "S0131"
    )
    .select(
        "shipment_id",
        "current_status",
        "current_status_time",
        "created_at",
        "packed_at",
        "shipped_at",
        "hub_scan_at",
        "out_for_delivery_at",
        "delivered_at",
        "is_delivered",
        "is_lifecycle_complete"
    )
)

In [0]:
print("=== Exemples de lifecycles complets ===")

display(
    shipment_lifecycle
    .filter(
        F.col("is_lifecycle_complete") == True
    )
    .select(
        "shipment_id",
        "current_status",
        "created_at",
        "packed_at",
        "shipped_at",
        "hub_scan_at",
        "out_for_delivery_at",
        "delivered_at",
        "delivery_duration_hours",
        "delivery_delay_hours"
    )
    .limit(10)
)